In [0]:
%run ../00-common/01.environment-config

In [0]:
# SOURCE AND TARGET PATHS

source_path = f"{s3_root_path}/Bronze/Aaisha/drivers"
target_path = f"{s3_root_path}/Silver/Aaisha/drivers"

print("Source Path :", source_path)
print("Target Path :", target_path)

In [0]:
drivers_df = (
    spark.read
        .format("delta")
        .load(source_path)
)

display(drivers_df)

In [0]:
from pyspark.sql import functions as F

drivers_clean_df = (
    drivers_df
        .dropna(how="all")
        .dropDuplicates()
        .filter(F.col("driverId").isNotNull())
        .withColumn("driverId", F.trim(F.col("driverId")))
        .withColumn("givenName", F.trim(F.col("name.givenName")))
        .withColumn("familyName", F.trim(F.col("name.familyName")))
        .withColumn("nationality", F.trim(F.col("nationality")))
        .withColumn("dateOfBirth", F.trim(F.col("dateOfBirth")))
        .withColumn("url", F.trim(F.col("url")))
)

In [0]:
drivers_final_df = drivers_clean_df.select(
    "driverId",
    "givenName",
    "familyName",
    "dateOfBirth",
    "nationality",
    "url",
    "ingestion_timestamp",
    "source_file"
)

display(drivers_final_df)

In [0]:
(
    drivers_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
silver_drivers_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(silver_drivers_df)